# Does a reinforcement-learning agent rediscover blackjack basic strategy?
### A policy-level audit of tabular Monte-Carlo control

An agent learns blackjack from nothing but the win/loss at the end of each hand. *Basic strategy* —
the provably-optimal action in every situation — is our ground-truth answer key. This notebook audits
the **learned policy against that key**: not whether the agent wins at the right rate, but **where its
decisions differ from optimal, and why** — and then *proves* the diagnosis with a forced-coverage
experiment (exploring starts).

**The finding, up front.** The agent reaches near-optimal play and reproduces the large majority of
basic strategy. The disagreements concentrate in the rarest, closest decisions (soft doubles, low
pairs). Most of that residual is the **cost of learning from experience** — situations rarely
experienced are rarely learned — which the capstone confirms by forcing coverage and watching the
residual collapse to a near-tie floor.

**Method discipline.** Agreement is reported visit-weighted *and* unweighted; disagreements are
separated by *severity*, not merely counted; runs are ranked only on estimates whose error bars
don't overlap.

## 1. Setup

In [ ]:
import json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
%matplotlib inline

RUNS = "../runs"

def pick(**crit):
    """Newest saved run matching the given config fields (and optional record-level ``method``)."""
    method = crit.pop("method", "__any__")
    def norm(cfg, k):
        v = cfg.get(k)
        if k == "with_splits":
            return bool(v)
        if k == "epsilon_schedule":
            return v or "constant"      # pre-schedule runs store None
        return v
    best = None
    for p in sorted(glob.glob(f"{RUNS}/*/record.json")):
        rec = json.load(open(p)); cfg = rec["config"]
        if method != "__any__" and rec.get("method") != method:
            continue
        if all(norm(cfg, k) == v for k, v in crit.items()):
            best = p
    if best is None:
        raise FileNotFoundError(f"no run matches {crit} method={method}")
    return json.load(open(best))

def cells_of(record):
    df = pd.DataFrame(record["diff"]["cells"])
    if "can_split" not in df:
        df["can_split"] = False
    df["can_split"] = df["can_split"].fillna(False).astype(bool)
    df["ev_gap"] = (df["agent_q"] - df["basic_q"]).abs()
    df["kind"] = ["pair" if c else ("soft" if s else "hard") for c, s in zip(df.can_split, df.is_soft)]
    return df

def run_label(record):
    if record.get("method") == "exploring_starts":
        return "exploring starts" + (", +splits" if record["config"].get("with_splits") else "")
    c = record["config"]
    sched = c.get("epsilon_schedule") or "constant"
    eps = (f"eps {c['epsilon']}" if sched == "constant"
           else f"{sched} {c.get('epsilon_start')}->{c.get('epsilon_end')}")
    alpha = f", a={c['step_size']}" if c.get("step_size") else ""
    splits = ", +splits" if c.get("with_splits") else ""
    return f"{eps}{alpha}{splits}"

baseline = pick(epsilon=0.1,  epsilon_schedule="constant", with_splits=False, method=None)
eps03    = pick(epsilon=0.3,  epsilon_schedule="constant", with_splits=False, method=None)
eps005   = pick(epsilon=0.05, epsilon_schedule="constant", with_splits=False, method=None)
decay    = pick(epsilon_schedule="linear", step_size=0.001, with_splits=False, method=None)
split    = pick(with_splits=True, method=None)
es       = pick(method="exploring_starts")
for nm, r in [("baseline",baseline),("eps0.3",eps03),("eps0.05",eps005),
              ("decay+a",decay),("split",split),("ES",es)]:
    print(f"{nm:9s} -> {run_label(r)}")

In [ ]:
PAIR_LABEL = {(4,False):"2,2",(6,False):"3,3",(8,False):"4,4",(10,False):"5,5",
              (12,False):"6,6",(14,False):"7,7",(16,False):"8,8",(18,False):"9,9",
              (20,False):"T,T",(12,True):"A,A"}
PAIR_ORDER = ["2,2","3,3","4,4","5,5","6,6","7,7","8,8","9,9","T,T","A,A"]

def plot_policy_heatmap(df, title=""):
    CATS = ["agree","near_equal_ev","genuine_disagreement","under_visited"]
    COLORS = {"agree":"#cdebc5","near_equal_ev":"#fde08a","genuine_disagreement":"#f3a3a3","under_visited":"#d9d9d9"}
    cat_i = {c:i for i,c in enumerate(CATS)}
    cmap = ListedColormap([COLORS[c] for c in CATS])
    ACT = {"hit":"H","stand":"S","double":"D","split":"P","surrender":"R"}
    ups = list(range(2,12))
    def grid(ax, d, sub, field, order):
        rows = [r for r in order if r in set(d[field])]
        g = np.full((len(rows), len(ups)), np.nan)
        for _, r in d.iterrows():
            if r[field] not in rows: continue
            i, j = rows.index(r[field]), ups.index(r["dealer_upcard"])
            g[i,j] = cat_i[r["category"]]
            lab = ACT[r["agent_action"]]
            if r["agent_action"] != r["basic_action"]:
                lab = f"{ACT[r['agent_action']]}\u2192{ACT[r['basic_action']]}"
            ax.text(j, i, lab, ha="center", va="center", fontsize=7)
        ax.imshow(g, cmap=cmap, vmin=0, vmax=len(CATS)-1, aspect="auto")
        ax.set_xticks(range(len(ups))); ax.set_xticklabels(ups)
        ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
        ax.set_xlabel("dealer upcard"); ax.set_title(sub)
    nonpair = df[~df["can_split"]]
    panels = [(nonpair[~nonpair["is_soft"]], "Hard totals", "player_value",
               sorted(nonpair[~nonpair["is_soft"]]["player_value"].unique())),
              (nonpair[nonpair["is_soft"]], "Soft totals", "player_value",
               sorted(nonpair[nonpair["is_soft"]]["player_value"].unique()))]
    if df["can_split"].any():
        pr = df[df["can_split"]].copy()
        pr["pair"] = [PAIR_LABEL.get((int(v),bool(s)), str(v)) for v,s in zip(pr.player_value, pr.is_soft)]
        panels.append((pr, "Pairs", "pair", [p for p in PAIR_ORDER if p in set(pr["pair"])]))
    fig, axes = plt.subplots(1, len(panels), figsize=(6.2*len(panels), 7))
    axes = np.atleast_1d(axes)
    for ax,(d,nm,f,o) in zip(axes, panels):
        grid(ax, d, nm, f, o)
    axes[0].set_ylabel("player total / pair")
    fig.legend(handles=[Patch(facecolor=COLORS[c], label=c) for c in CATS], loc="lower center", ncol=4)
    fig.suptitle(f"Learned policy vs basic strategy  {title}".strip())
    plt.tight_layout(rect=[0,0.05,1,1]); plt.show()

## 2. The question, and the method

The agent is a tabular Monte-Carlo control learner: it plays full hands, observes only the terminal
win/loss, and updates an action-value estimate `Q(state, action)` for every decision. Its *greedy*
policy is what we audit; basic strategy is the computed optimum, so disagreement means the agent
falling short of a known answer.

Each cell (player total x dealer upcard) is classified: **agree**, **near-equal-EV** (a true tie),
**genuine disagreement** (well-visited, different value), or **under-visited** (too few samples).

**A common trap, avoided.** This comparison is computed from the agent's **trained Q-table and its
training visit counts**, not from the separate house-edge evaluation. A cell's label is a property of
the trained policy and does not shift if we change the size of the edge evaluation.

## 3. The finding is localization, not the aggregate

"92.5% agreement" is true and nearly useless: it pools decisions the agent nails with the few it
struggles with. The structure appears the moment we *condition* on the type of decision.

In [ ]:
b = cells_of(baseline)
b["involves_double"] = b["agent_action"].eq("double") | b["basic_action"].eq("double")
genuine = lambda s: (s["category"] == "genuine_disagreement").mean()
print(f"pooled agreement (visit-weighted): {baseline['diff']['agreement_weighted']:.1%}")
print(f"pooled agreement (unweighted):     {(b['category']=='agree').mean():.1%}\n")
print("P(genuine disagreement) by decision type:")
print(f"  hard totals      : {genuine(b[~b.is_soft & ~b.can_split]):.1%}")
print(f"  soft totals      : {genuine(b[b.is_soft & ~b.can_split]):.1%}")
print(f"  doubling in play : {genuine(b[b.involves_double]):.1%}")

Disagreement is ~2% on hard totals, ~12% on soft, and ~27% wherever doubling is an option. The
agent has essentially learned hard-total play; its gaps live in the soft and doubling decisions —
the situations that arise least often. This localization, not the pooled number, is the result.

## 4. The disagreements — and their severity

A single hard threshold (`ev_gap > 0.02`) creates two traps: a **knife-edge** on the most important
cell — hard 16 vs 10 (~166k visits) sits at `ev_gap ~ 0.021`, a thousandth over the line, on the most
famously-balanced decision in the game — and a **severity-blind bucket** (genuine gaps span 20x,
0.021 to 0.407). So we report the distribution and a severity tier, not a count.

In [ ]:
gd = b[b.category == "genuine_disagreement"].copy()
gd["severity"] = pd.cut(gd["ev_gap"], [0,0.05,0.15,1.0], labels=["near-tie (<0.05)","moderate","severe (>0.15)"])
print("genuine disagreements:", len(gd))
print(gd["ev_gap"].describe()[["min","25%","50%","75%","max"]].round(3).to_string(), "\n")
print(gd["severity"].value_counts().sort_index().to_string())

A cluster of near-ties (incl. hard-16-vs-10), a body of moderate gaps, and a short tail. One
caution the severity view makes visible: a *large* gap usually means the agent's alternative action
was badly under-sampled and thus mis-valued — so large gaps are often a coverage symptom, which the
next section tests directly.

## 5. Mechanism — exploration starvation vs over-valuation

A wrong greedy action comes from the right action being **under-explored** (tried too rarely to be
valued) or a wrong action **over-valued** by lucky early returns. Counting how often each action was
tried in the disagreement cells separates them.

In [ ]:
qt = pd.DataFrame(baseline["qtable"])
def tries(row, action):
    m = ((qt.player_value==row.player_value)&(qt.is_soft==row.is_soft)
         &(qt.dealer_upcard==row.dealer_upcard)&(qt.action==action))
    if "can_split" in qt: m &= (qt.can_split==row.can_split)
    s = qt[m]
    return int(s["n"].iloc[0]) if len(s) else 0
mech = gd.copy()
mech["n_agent"] = [tries(r, r.agent_action) for r in gd.itertuples()]
mech["n_basic"] = [tries(r, r.basic_action) for r in gd.itertuples()]
mech["basic_share"] = (mech["n_basic"]/(mech["n_agent"]+mech["n_basic"])).round(2)
mech.sort_values("basic_share")[["player_value","is_soft","dealer_upcard","agent_action",
    "basic_action","n_agent","n_basic","basic_share","ev_gap"]].round(3)

Where `basic_share` is low, the *correct* action was barely tried — the agent never gathered the
evidence to value it. This is exploration starvation, dominant in the doubling cells. The fix is not
"explore more everywhere" (next section) but to schedule exploration and weight recent experience.

## 6. The exploration-bias tradeoff: the residual *relocates*

Raising epsilon does **not** shrink the disagreement — it **moves** it. We compare, cell by cell,
whether the agent matched basic at epsilon=0.1 versus epsilon=0.3.

In [ ]:
def match_series(rec):
    d = cells_of(rec); d["match"] = d.agent_action == d.basic_action
    return d.set_index(["player_value","is_soft","dealer_upcard"])["match"]
m1, m3 = match_series(baseline), match_series(eps03)
common = m1.index.intersection(m3.index)
worse  = int((m1.loc[common] & ~m3.loc[common]).sum())
better = int((~m1.loc[common] & m3.loc[common]).sum())
print(f"shared cells: {len(common)}")
print(f"  matched at eps0.1 but NOT eps0.3 (worse): {worse}")
print(f"  matched at eps0.3 but NOT eps0.1 (better): {better}")
print(f"  net: {better-worse:+d}  -> the error RELOCATES, not resolves")

More exploration buys coverage of rare cells at the cost of biasing the common ones (an on-policy
value reflects a behavior policy that now plays badly 30% of the time). Roughly as many cells get
worse as better — no single fixed epsilon wins everywhere, motivating a schedule plus
recency-weighting.

## 7. Why decay alone isn't enough — recency-weighting

Decaying epsilon toward zero seems obvious, but with a **sample-average** update (1/N) early high-eps
returns are baked in permanently — the average can't forget. The cure is a **constant step size
alpha** (recency-weighting). The ledger re-evaluates every saved run at the same large sample, so the
edges are comparable.

In [ ]:
from blackjack_rl.experiment import load_agent
from blackjack_rl.evaluation.metrics import evaluate_policy, GreedyPolicy

EVAL_N = 1_000_000   # same sample for every run. Slow (~minutes/run); lower to iterate.
seen = {}
for p in sorted(glob.glob(f"{RUNS}/*/record.json")):
    rec = json.load(open(p)); c = rec["config"]
    sig = (rec.get("method"), c["epsilon"], c.get("epsilon_schedule") or "constant",
           c.get("epsilon_start"), c.get("epsilon_end"), c.get("step_size"),
           bool(c.get("with_splits")), c["num_episodes"])
    seen[sig] = rec                       # newest wins (no duplicate rows, no redundant eval)
rows = []
for rec in seen.values():
    res = evaluate_policy(GreedyPolicy(load_agent(rec)), n_hands=EVAL_N, seed=0)
    rows.append({"run": run_label(rec), "episodes": rec["config"]["num_episodes"],
                 "edge_%": round(res.edge*100,3), "se_%": round(res.std_error*100,3),
                 "agree_wt": round(rec["diff"]["agreement_weighted"],3)})
ledger = pd.DataFrame(rows).sort_values("edge_%").reset_index(drop=True)
ledger

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(ledger["run"], ledger["edge_%"], xerr=ledger["se_%"], color="#6b9bd1",
        error_kw={"ecolor":"#444","capsize":3})
ax.invert_yaxis(); ax.set_xlabel("house edge %  (lower = better)")
ax.set_title(f"House edge by run  ({EVAL_N:,}-hand eval)")
plt.tight_layout(); plt.show()

Read against the error bars (~±0.11 each), the disciplined reading is: **two differences are real
(~2 sigma+):** adding splits helps, and **recency-weighting beats the same schedule with a plain
sample-average** — the constant-alpha payoff. But among the no-split runs, decay+α is only
*numerically* ahead of eps=0.1 and eps=0.05 (within ~1 sigma) — no winner there. High exploration
(eps=0.3, undamped sample-average) is clearly worst. (At 200k hands the top of this table inverted —
rank only on estimates whose error bars don't overlap.)

## 8. Convergence — how much training is enough

*Policy churn* (cells whose greedy action changed since the last checkpoint) shows when learning
stops; *minimum state-visits* tracks the worst-covered cell — the coverage floor.

In [ ]:
lc = pd.DataFrame(split["learning_curve"])
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(lc.episode/1e6, lc.policy_churn, marker="o", ms=3)
ax[0].set_xlabel("episodes (millions)"); ax[0].set_ylabel("policy churn"); ax[0].set_title("Convergence")
ax[1].plot(lc.episode/1e6, lc.min_state_visits, marker="o", ms=3, color="C3")
ax[1].set_xlabel("episodes (millions)"); ax[1].set_ylabel("min state visits"); ax[1].set_title("Worst-covered state")
plt.tight_layout(); plt.show()

Churn falls to a low plateau — the policy stops changing well before the last episode. But the
worst-covered state bottoms out at a handful of visits: even after millions of hands, the rarest cells
are barely seen. That floor is the coverage problem, quantified — the through-line to the splits and
the capstone.

## 9. Completing Problem A — splits

Pairs were added by *extending the state* (a `can_split` bit) and retraining — the agent discovers
splitting from experience, not from rules. The heatmap now carries a third panel for pairs.

In [ ]:
plot_policy_heatmap(cells_of(split), "— complete A (decay + alpha, with splits)")

In [ ]:
s = cells_of(split)
dis = s[s.category != "agree"].copy()
print(f"disagreements: {len(dis)} of {len(s)} cells")
print(dis.groupby(["category","kind"]).size().to_string())

Adding splits **improved the edge** (correct splitting recovers EV). The agent learned the bulk
of the split column; its misses are mostly **under-splitting the rare low pairs** (2,2 / 3,3 / 4,4)
and 9,9 — cells seen only ~2,300 times, where the split action itself was tried far fewer. Same
coverage signature as the doubling cells, in a new column.

## 10. The residual, honestly — the cost of learning from experience

The best natural-play agent approaches but does not match the optimum, and the gap concentrates in the
soft-double and low-pair cells. It decomposes into two parts:

- **Coverage (reducible).** Rare cells are rarely experienced, so their under-sampled actions stay
  mis-valued. This is the dominant part — the cost of learning from experience: basic strategy is
  computed exhaustively; the agent only knows what it has *seen*.
- **Near-ties (an irreduciable floor).** A few disagreements are genuine ties — the two actions differ in value by almost nothing (e.g. hard-16-vs-10). No amount of data separates them, because there is almost nothing to separate. This is not a representation limit: the optimum (basic strategy) is itself a table over these exact features, so the state is sufficient — the floor is the tie, not the abstraction.

The next section tests this split directly.

## 11. Capstone — exploring starts: is the residual really coverage?

If the residual is coverage, then *forcing* coverage should remove it. **Exploring starts** begins
every episode from a deliberately chosen (state, action) pair — every pair gets equal airtime
regardless of how rare it is in natural play — then follows the greedy policy (no epsilon). It is the
canonical MC-control variant (Sutton & Barto).

**The prediction, locked before the run:** the coverage-driven disagreements (under-split rare pairs,
under-explored doubles) should collapse toward basic strategy and the edge should move toward optimal;
the genuine **near-ties** should remain — an irreducible floor, not a coverage problem. If
*everything* vanished, that would be suspicious.

In [ ]:
es_d = cells_of(es); sp_d = cells_of(split)
def line(name, rec, d):
    g = d[d.category=="genuine_disagreement"]
    print(f"{name:20s} agree(unwt) {(d.category=='agree').mean():.3f}  "
          f"genuine {len(g):2d}  near_tie {(d.category=='near_equal_ev').sum():2d}  "
          f"under_visited {(d.category=='under_visited').sum():2d}  "
          f"max genuine ev_gap {g.ev_gap.max():.3f}")
line("decay+a, +splits", split, sp_d)
line("exploring starts", es, es_d)
print("\ngenuine disagreements by kind:")
print(" decay+a:", sp_d[sp_d.category=='genuine_disagreement'].kind.value_counts().to_dict())
print(" ES     :", es_d[es_d.category=='genuine_disagreement'].kind.value_counts().to_dict())

Genuine disagreements collapse (30 -> 9), the soft and pair components most of all — and the
*maximum* genuine ev_gap drops from ~0.4 to ~0.06. The large, coverage-driven misses are gone; only
small gaps survive.

In [ ]:
def act(d, pv, up, soft=False, csp=True):
    r = d[(d.player_value==pv)&(d.is_soft==soft)&(d.dealer_upcard==up)&(d.can_split==csp)]
    return "-" if r.empty else f"{r.iloc[0].agent_action}->{r.iloc[0].basic_action}"
rows = [{"pair":nm, "vs":up, "decay+a": act(sp_d,pv,up), "ES": act(es_d,pv,up)}
        for pv,up,nm in [(4,3,"2,2"),(6,4,"3,3"),(8,5,"4,4"),(18,3,"9,9"),(4,6,"2,2"),(6,6,"3,3")]]
pd.DataFrame(rows)

The previously under-split rare pairs flip to **split** under forced coverage — exactly the cells
the natural-play agent starved.

In [ ]:
plot_policy_heatmap(es_d, "— exploring starts (forced coverage)")

In [ ]:
g = es_d[es_d.category=="genuine_disagreement"].copy()
g["cell"] = [("pair "+PAIR_LABEL.get((int(v),bool(s)),str(v))) if c else (("soft " if s else "hard ")+str(v))
             for v,s,c in zip(g.player_value, g.is_soft, g.can_split)]
print("remaining genuine disagreements under ES (all near-ties):")
print(g.sort_values("ev_gap")[["cell","dealer_upcard","visits","agent_action","basic_action","ev_gap"]].round(3).to_string(index=False))
uv = es_d[es_d.category=="under_visited"]
print(f"\nunder-visited cells (2-card ES doesn't seed deep multi-card states): {len(uv)}")
print(list(zip(uv.player_value.tolist(), uv.dealer_upcard.tolist())))

**Verdict — the prediction held.** Forcing coverage collapsed the genuine disagreements from 30
to 9, and every survivor is a **near-tie** (ev_gap <= ~0.06) — the irreducable floor we predicted
would remain. The under-split rare pairs and the under-explored doubles flipped to basic strategy, and
on the ledger ES sits at/near basic strategy, below every natural-play run. The residual *was* mostly
coverage — "the cost of learning from experience" — and what's left is the genuinely-tied decisions
plus a handful of thin deep states.

**Two honest caveats.** (1) The edge claim rests on the 1M ledger re-eval, not the noisy 200k number
in the run record. (2) Exploring starts forces *2-card* start states, so a few deep multi-card hard
states stay thin (the `under_visited` cells above) — coverage is near-uniform over decision starts,
not literally every reachable state. ES earned the claim; it did not manufacture a clean sweep.

## Findings

1. **The agent rediscovers basic strategy where it matters** — near-optimal edge, hard-total play
   essentially solved.
2. **Errors are localized, not diffuse** (~2% hard vs ~27% doubling). Report localization, not pooled
   agreement.
3. **Severity matters.**  Reported as a distribution, not a count: most disagreements are near-ties (incl. hard-16-vs-10), and the few real ones traced to under-sampled actions — a coverage symptom, not confident-but-wrong play.
4. **More exploration relocates the residual, it doesn't remove it** — the core fixed-epsilon tradeoff.
5. **Recency-weighting (decay + constant-alpha) wins — modestly.** Rank only on tight estimates.
6. **Splits show the same story in a new column** — most learned, the rare pairs under-split.
7. **Exploring starts proves the diagnosis.**  Forcing coverage collapses the coverage residual (30→9 genuine) to a near-tie floor and moves the edge to basic strategy — the residual was mostly the cost of learning from experience (a few deep multi-card states stay thin, since forced starts seed 2-card hands).